# AP CSA Unit 2 – JWT Login with Selection and Iteration

---

## Learning Objectives

- Understand how a browser sends a JWT to a server in an authentication cookie.
- Use iteration to search a collection of cookies for an authentication token.
- Use selection to choose the correct authentication result.
- Trace how different inputs change a selection-and-iteration algorithm's output.
- Practice the algorithm with runnable demos, popcorn hacks, and a homework challenge.

---

## Key Vocabulary

| Term | Definition | Example |
|---|---|---|
| **JWT** | A signed token that carries claims between systems. | `abc.def.sig` |
| **Cookie** | A small name-value pair a browser can send with a request. | `auth_token=abc.def.sig` |
| **Iteration** | Repeating code for each item in a collection. | `for (String cookie : cookies)` |
| **Selection** | Choosing which path runs based on a condition. | `if (expired)` |
| **Guard clause** | An early check that rejects an invalid condition. | `if (jwt == null) return ...;` |
| **Authentication** | Checking whether a request comes from an accepted identity. | Allow or reject a request |

---

## Concept Overview

### How Does a JWT Cookie Move Through a Login?

After a successful login, a server can send a signed JWT in a cookie. On a later request, the browser sends that cookie back. The server searches for the authentication cookie and then checks the token before allowing access.

A server might send this response header after login:

```text
Set-Cookie: auth_token=<signed-jwt>; HttpOnly; Secure; SameSite=Lax
```

For this lesson, the web framework has already separated the incoming cookie header into a `String[]`. The Java algorithm has two jobs:

| Input | Processing | Output |
|---|---|---|
| Request cookies | Iterate until `auth_token` is found | JWT text or `null` |
| JWT validation results | Select the first matching rejection condition | `401` or `200` response |

You do not need to implement cryptography here. Treat `signatureValid` and `expired` as results already produced by a JWT library, and focus on the Java control flow.

---

## Iteration: Find the Authentication Cookie

The enhanced `for` loop examines one cookie at a time. When a cookie starts with `auth_token=`, the method returns the text after that prefix. Returning immediately stops the loop because the answer has been found. If every item is checked without a match, the method returns `null`.

In [ ]:
// Iteration Example - Search each cookie for the JWT.
public class CookieSearch {
    public static String findAuthToken(String[] cookies) {
        if (cookies == null) {
            return null;
        }

        for (String cookie : cookies) {
            if (cookie != null && cookie.startsWith("auth_token=")) {
                return cookie.substring("auth_token=".length());
            }
        }
        return null;
    }

    public static void main(String[] args) {
        String[] present = {"theme=dark", "auth_token=abc.def.sig"};
        String[] missing = {"theme=dark", "language=en"};

        System.out.println(findAuthToken(present));
        System.out.println(findAuthToken(missing));
    }
}
CookieSearch.main(null);

---

## Selection: Choose an Authentication Result

Ordered `if` statements reject one problem at a time. The order matters: there is no token to validate when the cookie is missing, an invalid signature must never be accepted, and an expired token must be rejected even if its signature is valid.

In [ ]:
// Selection Example - Choose the first matching result.
public class JwtSelection {
    public static String authorize(
            String jwt, boolean signatureValid, boolean expired) {
        if (jwt == null) {
            return "401: missing authentication cookie";
        }
        if (!signatureValid) {
            return "401: invalid JWT signature";
        }
        if (expired) {
            return "401: expired JWT";
        }
        return "200: request allowed";
    }

    public static void main(String[] args) {
        System.out.println(authorize(null, true, false));
        System.out.println(authorize("abc.def.sig", false, false));
        System.out.println(authorize("abc.def.sig", true, true));
        System.out.println(authorize("abc.def.sig", true, false));
    }
}
JwtSelection.main(null);

---

## Combining Iteration and Selection

A request handler can call the cookie-search method first and pass its result into the selection method. Keeping the two jobs in separate methods makes each method easier to read and test.

In [ ]:
// Complete Request Example
public class JwtCookieFlow {
    public static String findAuthToken(String[] cookies) {
        if (cookies == null) return null;

        for (String cookie : cookies) {
            if (cookie != null && cookie.startsWith("auth_token=")) {
                return cookie.substring("auth_token=".length());
            }
        }
        return null;
    }

    public static String authorize(
            String[] cookies, boolean signatureValid, boolean expired) {
        String jwt = findAuthToken(cookies);

        if (jwt == null) return "401: missing authentication cookie";
        if (!signatureValid) return "401: invalid JWT signature";
        if (expired) return "401: expired JWT";
        return "200: request allowed";
    }

    public static void main(String[] args) {
        String[] validCookies = {"theme=dark", "auth_token=abc.def.sig"};
        String[] missingCookies = {"theme=dark", "language=en"};

        System.out.println(authorize(validCookies, true, false));
        System.out.println(authorize(missingCookies, true, false));
        System.out.println(authorize(validCookies, false, false));
        System.out.println(authorize(validCookies, true, true));
    }
}
JwtCookieFlow.main(null);

---

## [Popcorn Hack] Practice #1: Trace the Loop

Use this cookie array:

```java
String[] cookies = {"theme=dark", "language=en", "auth_token=abc.def.sig"};
```

Answer before running the check below:

1. How many loop iterations occur before the method returns?
2. Which condition is `true` on the final iteration?
3. What value does the method return?

In [ ]:
// Practice #1 Answer
System.out.println("Iterations: 3");
System.out.println("True condition: cookie.startsWith(\"auth_token=\")");
System.out.println("Returned value: abc.def.sig");

---

## [Popcorn Hack] Practice #2: Complete the Selection

Complete `validateJwt`. Return `"invalid signature"` when the signature is invalid, `"expired"` when the token is expired, and `"valid"` otherwise. Keep the checks in that order so every test prints `true`.

In [ ]:
// Practice #2 - Add the ordered if statements.
public class JwtSelectionPractice {
    public static String validateJwt(boolean signatureValid, boolean expired) {
        // TODO: return the required result for each condition.
        return "not implemented";
    }

    public static void main(String[] args) {
        System.out.println(validateJwt(false, false).equals("invalid signature"));
        System.out.println(validateJwt(true, true).equals("expired"));
        System.out.println(validateJwt(true, false).equals("valid"));
    }
}
JwtSelectionPractice.main(null);

---

## Homework Hack: Authorize a Request

Complete `isAuthorized` by combining iteration and selection:

1. Iterate through `cookies` to determine whether an `auth_token` cookie exists.
2. Return `true` only when the cookie exists, the signature is valid, and the JWT is not expired.
3. Handle a `null` cookie array safely.

Run the cell and make all five checks print `true`. Use only the fake token shown in the starter; never submit a real password, session cookie, or JWT.

In [ ]:
// Homework Hack - Combine iteration and selection.
public class AuthAlgorithmHomework {
    public static boolean isAuthorized(
            String[] cookies, boolean signatureValid, boolean expired) {
        boolean authCookiePresent = false;

        // TODO: if cookies is not null, iterate and update authCookiePresent.

        // TODO: combine all three authorization conditions.
        return false;
    }

    public static void main(String[] args) {
        String[] present = {"theme=dark", "auth_token=abc.def.sig"};
        String[] missing = {"theme=dark"};

        System.out.println(isAuthorized(present, true, false));
        System.out.println(!isAuthorized(missing, true, false));
        System.out.println(!isAuthorized(present, false, false));
        System.out.println(!isAuthorized(present, true, true));
        System.out.println(!isAuthorized(null, true, false));
    }
}
AuthAlgorithmHomework.main(null);

---

## Summary

In [ ]:
// Iteration searches for the cookie.
String[] summaryCookies = {"theme=dark", "auth_token=abc.def.sig"};
boolean authCookiePresent = false;
for (String cookie : summaryCookies) {
    if (cookie.startsWith("auth_token=")) {
        authCookiePresent = true;
    }
}

// Selection combines the required conditions.
boolean signatureValid = true;
boolean expired = false;
boolean authorized = authCookiePresent && signatureValid && !expired;
System.out.println(authorized);

### Remember

- Iteration checks each cookie until the authentication cookie is found.
- Selection rejects missing, invalid, and expired tokens.
- The order of guard clauses makes the algorithm safe and readable.
- A real server must use a maintained JWT library, verify the expected algorithm and claims, and use HTTPS.
- Never print or submit real credentials or token contents.

**You can now trace selection and iteration through a JWT cookie flow.**

---

## References

- College Board. (2025). [*AP Computer Science A Course and Exam Description*](https://apcentral.collegeboard.org/media/pdf/ap-computer-science-a-course-and-exam-description.pdf), Unit 2: Selection and Iteration.
- Jones, M., Bradley, J., & Sakimura, N. (2015). [*JSON Web Token (JWT)*](https://www.rfc-editor.org/rfc/rfc7519). RFC 7519. Internet Engineering Task Force.
- MDN contributors. [*Set-Cookie header*](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Headers/Set-Cookie). MDN Web Docs.